# GTF / 02: long into the fix, then reverse into the short

Protocol: [`003_claude_gtf02_preregistration.md`](../../discussions/Gotobi_Tokyo_Fix/003_claude_gtf02_preregistration.md). Engine 0.2.0 (includes correction C1).

**Rule.** On a gotobi day buy USD/JPY at the ask at 09:25 JST (00:25 UTC), sell at the bid at 09:55 JST and, on the same tick, sell again to open the GTF 01 short (12-pip stop, buy back at the ask at 10:30 JST). Each leg pays its own spread and commission.

| Window | Dates | Role |
|---|---|---|
| EXPLORATORY-A | 2010–2015 | gates reported (already seen) |
| EXPLORATORY-B | 2016–2025 | gates reported; part of the verdict |
| HOLDOUT | 2026-01 → last downloaded day | **binding, one look**, opened only with `OPEN_HOLDOUT = True` |

Verdict `EDGE_CANDIDATE` needs G1–G3 on EXPLORATORY-B and HG1–HG3 on the holdout. `GTF_SYNTHETIC=1` runs offline (outputs to `02_synthetic/`).

In [ ]:
OPEN_HOLDOUT = False      # set True ONCE, after the exploratory sections have been run and committed

import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("GTF_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = tempfile.mkdtemp(prefix="gtf02_")
    os.environ.pop("DUKASCOPY_DIR", None)
    OPEN_HOLDOUT = True

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import gotobi as G

cfg = G.GTFConfig()
WINDOWS = {"EXPLORATORY-A": G.WINDOWS["INDEPENDENT"], "EXPLORATORY-B": G.WINDOWS["REPLICATION"]}
if SYNTHETIC:
    WINDOWS = {"EXPLORATORY-A": (dt.date(2014, 1, 1), dt.date(2014, 12, 31)),
               "EXPLORATORY-B": (dt.date(2016, 1, 1), dt.date(2016, 12, 31))}
    cfg = replace(cfg, n_boot=2_000, n_random_calendars=1_000)
START = min(w[0] for w in WINDOWS.values()); END = max(w[1] for w in WINDOWS.values())
OUT = Path.cwd() / ("02_synthetic" if SYNTHETIC else "02"); OUT.mkdir(exist_ok=True)
SHORT = G.grid_key(cfg.entry_min, cfg.exit_min, cfg.stop_pips)
LONG = f"long_e{cfg.long_entry_min}"
LONG_PX = f"long_price_e{cfg.long_entry_min}"
C35, C70 = cfg.commission_usd_per_m, 2 * cfg.commission_usd_per_m
JST = lambda m: f"{9 + m // 60:02d}:{m % 60:02d}"
pd.set_option("display.width", 200, "display.max_columns", 40)
print(f"gotobi {G.GOTOBI_VERSION} | synthetic={SYNTHETIC} | long {JST(cfg.long_entry_min)}->{JST(cfg.entry_min)} | "
      f"short {JST(cfg.entry_min)}->{JST(cfg.exit_min)} stop {cfg.stop_pips} | open_holdout={OPEN_HOLDOUT}")

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    G.write_synthetic(base, START, END)
    G.write_synthetic(base, dt.date(2026, 1, 1), dt.date(2026, 6, 30), seed=99)

## 1. Per-day outcomes (shared cache with notebook 01)

In [ ]:
root = G.dukascopy_root()
print("ticks:", root)

def prepare(days):
    days = days.copy()
    days["month_key"] = pd.to_datetime(days["date"]).dt.year * 12 + pd.to_datetime(days["date"]).dt.month
    both = days[LONG].notna() & days[SHORT].notna()
    days["long"] = days[LONG].where(both)
    days["short"] = days[SHORT].where(both)
    days["comb"] = G.combined_pnl(days, LONG, SHORT).where(both)
    days["long_net35"] = (days[LONG] - G.commission_pips(days[LONG_PX], C35)).where(both)
    days["comb_net35"] = G.combined_net(days, LONG, SHORT, LONG_PX, C35).where(both)
    days["comb_net70"] = G.combined_net(days, LONG, SHORT, LONG_PX, C70).where(both)
    return days

days = prepare(G.build_day_outcomes(START, END, cfg, root=root, n_jobs=4))
days["window"] = None
for w, (a, b) in WINDOWS.items():
    days.loc[(days["date"] >= a) & (days["date"] <= b), "window"] = w
print(days.groupby(["window", "day_type"])["comb"].count().unstack())

## 2. Mid path around the two legs

Mid minus the 09:55 mid. Shaded: the long leg (09:25–09:55) and the short leg (09:55–10:30).

In [ ]:
mins = list(cfg.path_minutes); cols = [f"mid_{m:03d}" for m in mins]
fig, axes = plt.subplots(1, len(WINDOWS), figsize=(13, 4), sharey=True)
for ax, w in zip(np.atleast_1d(axes), WINDOWS):
    d = days[days["window"] == w]
    rel = d[cols].sub(d["mid_fix"], axis=0) / G.PIP
    for t, style in [("gotobi", "-"), ("control", "--"), ("jp_holiday", ":")]:
        ax.plot(mins, rel[d["day_type"] == t].mean().to_numpy(), style, label=t)
    ax.axvspan(cfg.long_entry_min, cfg.entry_min, color="green", alpha=0.08)
    ax.axvspan(cfg.entry_min, cfg.exit_min, color="red", alpha=0.06)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xticks(range(0, 106, 15)); ax.set_xticklabels([JST(m) for m in range(0, 106, 15)])
    ax.set_title(w); ax.set_xlabel("JST"); ax.legend()
np.atleast_1d(axes)[0].set_ylabel("pips vs 09:55 mid")
plt.tight_layout(); plt.savefig(OUT / "02_mid_path.png", dpi=120); plt.show()

## 3. Exploratory gates

G1–G3 on the combined P&L, the long-leg check L, and the short alone (GTF 01 rule on the same days) for reference.

In [ ]:
def gate_block(d, col, net_col, net2_col):
    d = d.sort_values("date")
    g, c = d[d["day_type"] == "gotobi"], d[d["day_type"] == "control"]
    x = g[col].dropna().to_numpy()
    hac = G.hac_mean(x, cfg.hac_lags)
    boot = [G.block_bootstrap_lower(x, b, cfg.n_boot, cfg.alpha, cfg.seed) for b in cfg.blocks]
    diff = G.welch_diff(x, c[col])
    bank = d[d["jp_bank_day"]]
    rand = G.random_calendar_means(bank[col].to_numpy(), bank["month_key"].to_numpy(), bank["gotobi"].to_numpy(),
                                   cfg.n_random_calendars, cfg.seed)
    rank = G.placebo_rank(x.mean(), rand)
    net = g[net_col].dropna().to_numpy()
    boot_net = [G.block_bootstrap_lower(net, b, cfg.n_boot, cfg.alpha, cfg.seed) for b in cfg.blocks]
    net2 = float(g[net2_col].mean()) if net2_col else np.nan
    st = {"trades": int(len(x)), "mean": hac["mean"], "hac_t": hac["t"], "hac_p": hac["p_one_sided"],
          "win_rate": float((x > 0).mean()), "boot_lower": [round(b["lower"], 3) for b in boot],
          "control_mean": float(c[col].mean()), "diff": diff["diff"], "welch_p": diff["p_one_sided"],
          "random_cal_p": rank["p_upper"], "mean_net_c35": float(net.mean()),
          "boot_net_lower": [round(b["lower"], 3) for b in boot_net], "mean_net_c70": net2}
    return st, (hac, boot, diff, rank, boot_net, net2)

stats, gates, L = {}, {}, {}
for w in WINDOWS:
    d = days[days["window"] == w]
    st, parts = gate_block(d, "comb", "comb_net35", "comb_net70")
    stats[(w, "combined")] = st
    gates[w] = G.gates(w, *parts, cfg.alpha)
    stats[(w, "long 09:25-09:55")], _ = gate_block(d, "long", "long_net35", None)
    stats[(w, "short (GTF 01)")], _ = gate_block(d.assign(short_net35=G.net_pips(d["short"], d["entry_price"], C35)),
                                               "short", "short_net35", None)
    lg = d.loc[d["day_type"] == "gotobi"].sort_values("date")["long_net35"].dropna().to_numpy()
    lh = G.hac_mean(lg, cfg.hac_lags)
    L[w] = {"long_net35_mean": lh["mean"], "hac_p": lh["p_one_sided"],
            "L_pass": bool(lh["mean"] > 0 and lh["p_one_sided"] <= cfg.alpha)}
stat_tab = pd.DataFrame(stats).T
display(stat_tab)
display(pd.DataFrame(gates).T); display(pd.DataFrame(L).T)
stat_tab.to_csv(OUT / "02_exploratory_stats.csv")
pd.DataFrame(gates).T.to_csv(OUT / "02_exploratory_gates.csv"); pd.DataFrame(L).T.to_csv(OUT / "02_long_leg_L.csv")

## 4. By year

In [ ]:
g_all = days[(days["day_type"] == "gotobi") & days["window"].notna()]
by_year = g_all.groupby("year").agg(trades=("comb", "count"), long=("long", "mean"), short=("short", "mean"),
                                    combined=("comb", "mean"), combined_net35=("comb_net35", "mean"))
by_year["control_combined"] = days[days["day_type"] == "control"].groupby("year")["comb"].mean()
display(by_year.round(2)); by_year.to_csv(OUT / "02_by_year.csv")
ax = by_year[["long", "short", "combined"]].plot.bar(figsize=(11, 3.5)); ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("pips/day after spread"); plt.tight_layout(); plt.savefig(OUT / "02_by_year.png", dpi=120); plt.show()

## 5. Long entry time (reported, not gated)

Long from each entry time to 09:55, alone and combined with the short. Gotobi mean and gotobi − control, after spread.

In [ ]:
rows = []
for w in WINDOWS:
    d = days[days["window"] == w]
    for e in cfg.long_grid_entry:
        both = d[f"long_e{e}"].notna() & d[SHORT].notna()
        for name, s in [("long", d[f"long_e{e}"]), ("combined", d[f"long_e{e}"] + d[SHORT])]:
            s = s.where(both)
            gm, cm = s[d["day_type"] == "gotobi"].mean(), s[d["day_type"] == "control"].mean()
            rows.append({"window": w, "long_entry": JST(e), "series": name, "gotobi": gm, "diff_vs_control": gm - cm})
grid = pd.DataFrame(rows)
display(grid.pivot_table(index=["series", "long_entry"], columns="window", values=["gotobi", "diff_vs_control"]).round(2))
grid.to_csv(OUT / "02_long_entry_grid.csv", index=False)

## 6. Costs and equity

Break-even commission for the combined trade (both legs). Equity: same notional on both legs, return = 0.005 · net / 12 (GTF 01 sizing), net at c = 35.

In [ ]:
cost_rows = []
for w in WINDOWS:
    g = days[(days["window"] == w) & (days["day_type"] == "gotobi")].dropna(subset=["comb"])
    per_c = (2 * (g[LONG_PX] + g["entry_price"]) / 10_000).mean()
    row = {"window": w, "combined_after_spread": g["comb"].mean(), "break_even_c": g["comb"].mean() / per_c}
    for c in (0, 18, 35, 70):
        for slip in (0.0, 0.2, 0.5):
            row[f"net c={c} slip={slip}"] = G.combined_net(g, LONG, SHORT, LONG_PX, c, slip).mean()
    cost_rows.append(row)
costs = pd.DataFrame(cost_rows).set_index("window").T
display(costs.round(2)); costs.to_csv(OUT / "02_costs.csv")

g = days[(days["day_type"] == "gotobi") & days["window"].notna()].dropna(subset=["comb"]).sort_values("date").set_index("date")
eq_c = G.equity_curve(g["comb_net35"]); eq_s = G.equity_curve(pd.Series(G.net_pips(g["short"], g["entry_price"], C35), index=g.index))
yrs = (pd.Timestamp(g.index[-1]) - pd.Timestamp(g.index[0])).days / 365.25
for name, eq in [("combined", eq_c), ("short only", eq_s)]:
    print(f"{name:10s} x{eq['equity'].iloc[-1]:.3f} -> {eq['equity'].iloc[-1] ** (1 / yrs) - 1:.2%}/yr, max DD {eq['drawdown'].min():.2%}")
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(pd.to_datetime(eq_c.index), eq_c["equity"], label="long + short"); ax.plot(pd.to_datetime(eq_s.index), eq_s["equity"], label="short only")
ax.set_ylabel("equity (net c=35)"); ax.legend(); plt.tight_layout(); plt.savefig(OUT / "02_equity.png", dpi=120); plt.show()

## 7. HOLDOUT (binding, one look)

Runs only with `OPEN_HOLDOUT = True`. Needs the 2026 download (`--start 2026-01-01 --hours 22-2`). The rule above is frozen; nothing is changed after this cell has been run.

In [ ]:
holdout = None
if OPEN_HOLDOUT:
    hdays = sorted(p.stem.split("_")[-1] for p in (root / "2026").glob(f"{G.SYMBOL}_2026-*.parquet"))
    if not hdays:
        raise FileNotFoundError("No 2026 files: run the downloader with --start 2026-01-01 first.")
    h_end = dt.date.fromisoformat(hdays[-1])
    hd = prepare(G.build_day_outcomes(dt.date(2026, 1, 1), h_end, cfg, root=root, n_jobs=4, allow_holdout=True))
    hd = hd.sort_values("date")
    g, c = hd[hd["day_type"] == "gotobi"], hd[hd["day_type"] == "control"]
    x = g["comb"].dropna().to_numpy()
    hac = G.hac_mean(x, cfg.hac_lags); diff = G.welch_diff(x, c["comb"])
    hg = G.holdout_gates(hac, diff, float(g["comb_net35"].mean()))
    holdout = {"end": str(h_end), "trades": int(len(x)), "combined_mean": hac["mean"], "hac_p": hac["p_one_sided"],
               "control_mean": float(c["comb"].mean()), "welch_p": diff["p_one_sided"],
               "combined_net35_mean": float(g["comb_net35"].mean()), "long_net35_mean": float(g["long_net35"].mean()),
               "short_mean": float(g["short"].mean()), "long_mean": float(g["long"].mean()), **hg}
    display(pd.Series(holdout))
    pd.Series(holdout).to_csv(OUT / "02_holdout.csv")
else:
    print("Holdout sealed (OPEN_HOLDOUT = False).")

## 8. Verdict

In [ ]:
b = gates["EXPLORATORY-B"]
expl_ok = b["G1_statistical"] and b["G2_calendar"] and b["G3_economic"]
summary = {"study": "GTF02", "gotobi_version": G.GOTOBI_VERSION, "synthetic": SYNTHETIC, "data_root": str(root),
           "rule": {"long": f"{JST(cfg.long_entry_min)}-{JST(cfg.entry_min)} JST", "short": f"{JST(cfg.entry_min)}-{JST(cfg.exit_min)} JST stop {cfg.stop_pips}"},
           "exploratory_gates": gates, "long_leg_L": L,
           "stats": {f"{w}|{s}": v for (w, s), v in stats.items()}, "holdout": holdout}
if holdout is None:
    summary["verdict"] = "PENDING_HOLDOUT" if expl_ok else "NO_EDGE"
else:
    hold_ok = holdout["HG1_statistical"] and holdout["HG2_calendar"] and holdout["HG3_economic"]
    summary["verdict"] = "EDGE_CANDIDATE" if expl_ok and hold_ok else "NO_EDGE"
    summary["long_leg"] = "LONG_ADDS_VALUE" if L["EXPLORATORY-B"]["L_pass"] and holdout["long_net35_mean"] > 0 else "LONG_NOT_SHOWN"
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("exploratory_gates", "long_leg_L", "verdict")} | ({"long_leg": summary["long_leg"]} if "long_leg" in summary else {}), indent=2, default=str))